# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = pd.read_csv('/content/gapminder.csv')

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:
# Task 1
# -------
asia = df.loc[df['continent'] == 'Asia']
asia_2 = asia[asia['year'].isin([2002, 2007])].copy()
asia_2['year_label'] = asia_2['year'].astype(str)  # treat year as a category, not a continuous scale

year_colors = {'2002': '#C9D6DF', '2007': '#2E86AB'}  # grey for the earlier year, blue for the later

fig1 = px.scatter(
    asia_2, x='gdpPercap', y='lifeExp',
    color='year_label',
    color_discrete_map=year_colors,
    category_orders={'year_label': ['2002', '2007']},
    log_x=True,
    hover_name='country',
)

avg_change = (asia_2[asia_2['year'] == 2007]['lifeExp'].mean()
              - asia_2[asia_2['year'] == 2002]['lifeExp'].mean())

fig1.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color='white')))

fig1.update_layout(
    title=dict(
        text=(f"<b>Life expectancy across Asia rose by {avg_change:.1f} years on average, 2002 to 2007</b>"
              f"<br><sup>GDP per capita (log scale) vs. life expectancy, Asian countries</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    xaxis=dict(title='GDP per capita (log scale, $)', showgrid=True, gridcolor='#eeeeee', zeroline=False),
    yaxis=dict(title='Life expectancy (years)', showgrid=True, gridcolor='#eeeeee', zeroline=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    legend=dict(title='Year', orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
    height=500,
    width=850,
    margin=dict(l=10, r=20, t=100, b=40),
)

fig1.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:
# Task 2
# -------
d2007 = df.loc[df['year'] == 2007].copy()

highlight = 'Africa'
grey = '#C9D6DF'
highlight_color = '#D55E00'

d2007['plot_color'] = d2007['continent'].apply(lambda c: highlight_color if c == highlight else grey)

fig2 = px.scatter(
    d2007, x='gdpPercap', y='lifeExp',
    size='pop', size_max=55,
    log_x=True,
    hover_name='country',
)
fig2.update_traces(marker=dict(color=d2007['plot_color'], line=dict(width=0.5, color='white'), opacity=0.85))

africa_life = d2007[d2007['continent'] == highlight]['lifeExp'].mean()
global_life = d2007['lifeExp'].mean()
gap = global_life - africa_life

# Annotate the story directly on the chart, pointing at Nigeria (Africa's largest bubble)
nigeria = d2007[d2007['country'] == 'Nigeria'].iloc[0]
fig2.add_annotation(
    x=nigeria['gdpPercap'], y=nigeria['lifeExp'],
    text=(f"<b>Africa</b>: life expectancy trails the global<br>average by {gap:.0f} years, "
          f"despite large, growing populations<br>like Nigeria's ({nigeria['pop']/1e6:.0f}M people)"),
    showarrow=True, arrowhead=2, ax=90, ay=-60,
    font=dict(size=12, color=highlight_color),
    align='left',
    bgcolor='white',
    bordercolor=highlight_color,
    borderwidth=1,
)

fig2.update_layout(
    title=dict(
        text=(f"<b>Africa lags {gap:.0f} years behind the global average in life expectancy, despite population growth</b>"
              f"<br><sup>GDP per capita (log scale) vs. life expectancy, bubble size = population, 2007</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    xaxis=dict(title='GDP per capita (log scale, $)', showgrid=True, gridcolor='#eeeeee', zeroline=False),
    yaxis=dict(title='Life expectancy (years)', showgrid=True, gridcolor='#eeeeee', zeroline=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    showlegend=False,
    height=550,
    width=900,
    margin=dict(l=10, r=20, t=100, b=40),
)

fig2.show()
